In [ ]:
map = {
    0: "GlobalEventID", 
    1: "date", 
    2: "MonthYear", 
    3: "Year", 
    4: "FractionDate", 
    5: "Actor1Code", 
    6: "Actor1Name", 
    7: "Actor1CountryCode", 
    8: "Actor1KnownGroupCode", 
    9: "Actor1EthnicCode",
    10: "Actor1Religion1Code", 
    11: "Actor1Religion2Code", 
    12: "Actor1Type1Code", 
    13: "Actor1Type2Code",
    14: "Actor1Type3Code", 
    15: "Actor2Code", 
    16: "Actor2Name", 
    17: "Actor2CountryCode", 
    18: "Actor2KnownGroupCode", 
    19: "Actor2EthnicCode",
    20: "Actor2Religion1Code", 
    21: "Actor2Religion2Code", 
    22: "Actor2Type1Code", 
    23: "Actor2Type2Code", 
    24: "Actor2Type3Code", 
    25: "IsRootEvent", 
    26: "EventCode", 
    27: "EventBaseCode", 
    28: "EventRootCode", 
    29: "QuadClass",
    30: "GoldsteinScale", 
    31: "NumMentions", 
    32: "NumSources", 
    33: "NumArticles", 
    34: "AvgTone", 
    35: "Actor1Geo_Type", 
    36: "Actor1Geo_Fullname", 
    37: "Actor1Geo_CountryCode", 
    38: "Actor1Geo_ADM1Code", 
    39: "Actor1Geo_Lat",
    40: "Actor1Geo_Long", 
    41: "Actor1Geo_FeatureID", 
    42: "Actor2Geo_Type", 
    43: "Actor2Geo_Fullname", 
    44: "Actor2Geo_CountryCode", 
    45: "Actor2Geo_ADM1Code", 
    46: "Actor2Geo_Lat",
    47: "Actor2Geo_Long",
    48: "Actor2Geo_FeatureID", 
    49: "ActionGeo_Type", 
    50: "ActionGeo_Fullname", 
    51: "ActionGeo_CountryCode", 
    52: "ActionGeo_ADM1Code", 
    53: "ActionGeo_Lat", 
    54: "ActionGeo_Long", 
    55: "ActionGeo_FeatureID",
    56: "DATEADDED", 
    57: "SOURCEURL"
}

In [ ]:
#d_200908 = pd.read_csv('200908.CSV',sep='\t',header=None)
d_20140523 = pd.read_csv('20140523.export.CSV',sep='\t',header=None)
d_20140524 = pd.read_csv('20140524.export.CSV',sep='\t',header=None)
##elliot rodgers CA
d_20160612 = pd.read_csv('20160612.export.CSV',sep='\t',header=None)
##orlando night club FL
d_20170812 = pd.read_csv('20170812.export.CSV',sep='\t',header=None)
##charlottesville VA
d_20171031 = pd.read_csv('20171031.export.CSV',sep='\t',header=None)
##truck attack NY
d_20180423 = pd.read_csv('20180423.export.CSV',sep='\t',header=None)
##TORONTO skip for now
d_20200526 = pd.read_csv('20200526.export.CSV',sep='\t',header=None)
##george floyd MN
d_20250101 = pd.read_csv('20250101.export.CSV',sep='\t',header=None)
##truck attack neworleans TN
d_20250602 = pd.read_csv('20250602.export.CSV',sep='\t',header=None)

In [ ]:

def clean(df):
    values_to_keep = [193,190,10,12,173,90,112,145,20,40,111,43,42,51,141,11,13,14,15,17,18,19]
    df = df.rename(columns=map)
    df = df[(df['Actor1Geo_CountryCode'] == 'US') | (df['Actor2Geo_CountryCode'] == 'US') | (df['ActionGeo_CountryCode'] == 'US')]
    df = df[df['EventCode'].isin(values_to_keep)]
    df = df.drop(columns=['MonthYear', 'FractionDate','Actor1Code', 'Actor1Name','Actor1CountryCode', 'Actor1KnownGroupCode','Actor1EthnicCode', 'Actor1Religion1Code','Actor1Religion2Code', 'Actor1Type1Code','Actor1Type2Code','Actor1Type3Code','Actor2Code', 'Actor2Name','Actor2CountryCode', 'Actor2KnownGroupCode','Actor2EthnicCode', 'Actor2Religion1Code','Actor2Religion2Code', 'Actor2Type1Code','Actor2Type2Code','Actor2Type3Code','Actor1Geo_Type', 'Actor1Geo_Fullname','Actor1Geo_ADM1Code', 'Actor1Geo_Lat','Actor1Geo_Long', 'Actor1Geo_FeatureID', 'Actor2Geo_Type', 'Actor2Geo_Fullname', 'Actor2Geo_ADM1Code', 'Actor2Geo_Lat','Actor2Geo_Long','Actor2Geo_FeatureID'], axis=1)
    df = df[(df['AvgTone'] < 1)]
    return df

def simplify(domain):
    parts = domain.split('.')
    # Remove common subdomain prefixes like 'www'
    if parts[0] == 'www':
        parts = parts[1:]
    # Heuristic: use the first part before the main TLD (.com, .org, etc.)
    return parts[-3] if len(parts) > 2 else parts[0]

In [ ]:
d_20140523_CA = d_20140523[(d_20140523['ActionGeo_FeatureID'] == 'CA')]
#102
d_20140524_CA= d_20140524[(d_20140524['ActionGeo_FeatureID'] == 'CA')]
#309
d_20160612_FL= d_20160612[(d_20160612['ActionGeo_FeatureID'] == 'FL')]
#2349
d_20170812_VA= d_20170812[(d_20170812['ActionGeo_FeatureID'] == 'VA')]
#1037
d_20171031_NY= d_20171031[(d_20171031['ActionGeo_FeatureID'] == 'NY')]
#2873 most likely half
#d_20180423= d_20180423[(d_20180423['ActionGeo_FeatureID'] == 'FL')]
d_20200526_MN = d_20200526[(d_20200526['ActionGeo_FeatureID'] == 'MN')]
#405 most likely more in coming days
d_20250101_TN = d_20250101[(d_20250101['ActionGeo_FeatureID'] == 'TN')]
##not viable
d_20250602_CO = d_20250602[(d_20250602['ActionGeo_FeatureID'] == 'CO')]
#1170

In [ ]:
def goose_it(df):
    for i in df.index:
        try:
            url = df.loc[i, 'SOURCEURL']
            article = g.extract(url=url)
            df.at[i, 'domain'] = article.domain
            df.at[i,'target'] = simplify(article.domain)
            df.at[i, 'title'] = article.title
            df.at[i, 'text'] = article.cleaned_text
            df.at[i, 'description'] = article.opengraph.get('description', '')
            
        except NetworkError:
            print(f"NetworkError: Dropping index {i} with URL: {df.loc[i, 'SOURCEURL']}")
            df.drop(index=i, inplace=True)
        except Exception as e:
            print(f"Error at index {i}: {e}")
            print(df.loc[i])
            df.drop(index=i, inplace=True)
    return df

In [ ]:
f_20140524_CA = goose_it(d_20140524_CA)
f_20160612_FL = goose_it(d_20160612_FL)
f_20170812_VA = goose_it(d_20170812_VA)
f_20171031_NY = goose_it(d_20171031_NY)
f_20250602_CO = goose_it(d_20250602_CO)

In [ ]:
mbfcs = pd.read_csv('data/mbfc_source_data.csv')
def replace_nans(text_list, replacement=""):
    return ["" if str(x).strip().lower() == "nan" or x != x else x for x in text_list]

def target(df, mbfcs):
    df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    df['factual_reporting'] = ''  # Initialize column as object type
    df['factual_reporting'] = df['factual_reporting'].astype('object')
    # Normalize mbfcs keys for matching
    mbfcs['normalized_display_name'] = mbfcs['display_name'].str.lower().str.replace(" ", "", regex=False)
    mbfcs['normalized_website'] = mbfcs['website'].apply(lambda x: urlparse(x).hostname if pd.notnull(x) else '').str.lower()

    for i in df.index:
        
        target_clean = str(df.loc[i, 'target']).lower().replace(" ", "")
        domain_clean = str(df.loc[i, 'domain']).lower().replace(" ", "")

        match = mbfcs[mbfcs['normalized_display_name'] == target_clean]
        if not match.empty:
            df.at[i, 'factual_reporting'] = match.iloc[0]['factual_reporting']
            continue

        match = mbfcs[mbfcs['normalized_website'] == domain_clean]
        if not match.empty:
            df.at[i, 'factual_reporting'] = match.iloc[0]['factual_reporting']
        else:
            df.at[i, 'factual_reporting'] = None
    df = df.dropna(axis=0, subset=['text'])
    return df


d_20171031_NY = target(d_20171031_NY, mbfcs)
d_20170812_VA = target(d_20170812_VA, mbfcs)
d_20160612_FL = target(d_20160612_FL, mbfcs)
d_20250602_CO = target(d_20250602_CO, mbfcs)
d_20140524_CA = target(d_20140523_CA, mbfcs)

result_series = set(d_20140524_CA.loc[d_20140524_CA['factual_reporting'].isnull(), 'domain']) | \
                set(d_20171031_NY.loc[d_20171031_NY['factual_reporting'].isnull(), 'domain'])| \
                set(d_20170812_VA.loc[d_20170812_VA['factual_reporting'].isnull(), 'domain'])| \
                set(d_20160612_FL.loc[d_20160612_FL['factual_reporting'].isnull(), 'domain'])| \
                set(d_20250602_CO.loc[d_20250602_CO['factual_reporting'].isnull(), 'domain'])
